In [ ]:
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless

!pip install opencv-contrib-python==4.10.0.84

In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

In [ ]:
print("OpenCV version:", cv2.__version__)

print("cv2.face available:", hasattr(cv2, "face"))

if hasattr(cv2, "face"):
    print("LBPH available:", hasattr(cv2.face, "LBPHFaceRecognizer_create"))

In [ ]:
BASE_DIR = "/content/face_authentication"

DATASET_DIR = os.path.join(BASE_DIR, "dataset")
MODEL_DIR = os.path.join(BASE_DIR, "model")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Dataset directory:", DATASET_DIR)
print("Model directory:", MODEL_DIR)

In [ ]:
USER_NAME = "Alejandro Zermeno"

USER_DIR = os.path.join(DATASET_DIR, USER_NAME)

os.makedirs(USER_DIR, exist_ok=True)

print("User:", USER_NAME)
print("Folder:", USER_DIR)

In [ ]:
FACE_CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"

face_detector = cv2.CascadeClassifier(FACE_CASCADE_PATH)

if face_detector.empty():
    print("Error loading face detector.")
else:
    print("Face detector loaded successfully.")

In [ ]:
def take_photo(filename="photo.jpg", quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');

            const video = document.createElement('video');
            video.style.display = 'block';

            const stream = await navigator.mediaDevices.getUserMedia({
                video: true
            });

            document.body.appendChild(div);
            div.appendChild(video);

            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(
                document.documentElement.scrollHeight,
                true
            );

            const canvas = document.createElement('canvas');

            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;

            canvas.getContext('2d').drawImage(
                video,
                0,
                0
            );

            stream.getVideoTracks()[0].stop();
            div.remove();

            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')

    display(js)

    data = eval_js(
        f'takePhoto({quality})'
    )

    binary = b64decode(
        data.split(',')[1]
    )

    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

In [ ]:
photo_path = take_photo()

img = cv2.imread(photo_path)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
photo_path = take_photo("test_face.jpg")

img = cv2.imread(photo_path)

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

faces = face_detector.detectMultiScale(
    gray,
    scaleFactor=1.3,
    minNeighbors=5
)

print("Faces detected:", len(faces))

for (x, y, w, h) in faces:
    cv2.rectangle(
        img,
        (x, y),
        (x + w, y + h),
        (255, 0, 0),
        2
    )

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
NUM_IMAGES = 30

print(f"Starting capture of {NUM_IMAGES} images...")

saved_images = 0

for i in range(NUM_IMAGES):

    filename = f"capture_{i+1}.jpg"
    temp_path = os.path.join(USER_DIR, filename)

    try:
        photo_path = take_photo(
            temp_path,
            quality=0.8
        )

        img = cv2.imread(photo_path)

        gray = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2GRAY
        )

        faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.3,
            minNeighbors=5
        )

        if len(faces) > 0:

            # Tomar el rostro más grande
            face = max(
                faces,
                key=lambda rect: rect[2] * rect[3]
            )

            x, y, w, h = face

            face_img = gray[
                y:y+h,
                x:x+w
            ]

            cv2.imwrite(
                temp_path,
                face_img
            )

            saved_images += 1

            print(
                f"Image {i+1}: face detected and saved."
            )

        else:

            if os.path.exists(temp_path):
                os.remove(temp_path)

            print(
                f"Image {i+1}: no face detected."
            )

    except Exception as e:

        print(
            f"Error on image {i+1}: {e}"
        )

print()
print("Images saved:", saved_images)

In [ ]:
image_files = sorted(
    [
        f for f in os.listdir(USER_DIR)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
)

print("Total images:", len(image_files))

fig, axes = plt.subplots(
    5,
    6,
    figsize=(12, 10)
)

axes = axes.flatten()

for i, filename in enumerate(image_files[:30]):

    img = cv2.imread(
        os.path.join(USER_DIR, filename),
        cv2.IMREAD_GRAYSCALE
    )

    axes[i].imshow(
        img,
        cmap="gray"
    )

    axes[i].set_title(
        filename
    )

    axes[i].axis("off")

for i in range(len(image_files), len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
faces = []
labels = []

label_id = 0

user_folders = [
    folder
    for folder in os.listdir(DATASET_DIR)
    if os.path.isdir(
        os.path.join(DATASET_DIR, folder)
    )
]

print("Users found:")
print(user_folders)

In [ ]:
for user_name in user_folders:

    user_path = os.path.join(
        DATASET_DIR,
        user_name
    )

    print(
        f"Loading images for: {user_name}"
    )

    image_files = [
        f
        for f in os.listdir(user_path)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

    for image_file in image_files:

        image_path = os.path.join(
            user_path,
            image_file
        )

        img = cv2.imread(
            image_path,
            cv2.IMREAD_GRAYSCALE
        )

        if img is not None:

            faces.append(img)
            labels.append(label_id)

    label_id += 1

print()
print("Total training images:", len(faces))
print("Total labels:", len(labels))

In [ ]:
if len(faces) == 0:

    print("ERROR: No training images were found.")

else:

    print("Dataset ready.")
    print("Images:", len(faces))
    print("Labels:", len(labels))

    print(
        "Image size:",
        faces[0].shape
    )

In [ ]:
recognizer = cv2.face.LBPHFaceRecognizer_create()

print("LBPH recognizer created successfully.")

In [ ]:
recognizer.train(
    faces,
    np.array(labels)
)

print("Training completed successfully.")

In [ ]:
MODEL_PATH = os.path.join(
    MODEL_DIR,
    "face_model.yml"
)

recognizer.write(
    MODEL_PATH
)

print(
    "Model saved at:",
    MODEL_PATH
)

In [ ]:
label_map = {}

for label, user_name in enumerate(user_folders):
    label_map[label] = user_name

print("Label map:")
print(label_map)

In [ ]:
import json

LABEL_MAP_PATH = os.path.join(
    MODEL_DIR,
    "label_map.json"
)

with open(
    LABEL_MAP_PATH,
    "w"
) as f:

    json.dump(
        label_map,
        f
    )

print(
    "Label map saved at:",
    LABEL_MAP_PATH
)

In [ ]:
recognizer = cv2.face.LBPHFaceRecognizer_create()

recognizer.read(
    MODEL_PATH
)

with open(
    LABEL_MAP_PATH,
    "r"
) as f:

    label_map = json.load(f)

label_map = {
    int(k): v
    for k, v in label_map.items()
}

print("Model loaded successfully.")
print("Authorized users:", label_map)

In [ ]:
test_path = take_photo(
    "recognition_test.jpg"
)

test_img = cv2.imread(
    test_path
)

gray = cv2.cvtColor(
    test_img,
    cv2.COLOR_BGR2GRAY
)

faces_detected = face_detector.detectMultiScale(
    gray,
    scaleFactor=1.3,
    minNeighbors=5
)

print(
    "Faces detected:",
    len(faces_detected)
)

In [ ]:
for (x, y, w, h) in faces_detected:

    face = gray[
        y:y+h,
        x:x+w
    ]

    prediction, confidence = recognizer.predict(
        face
    )

    print("Prediction:", prediction)
    print("Confidence:", confidence)

    if prediction in label_map:

        predicted_name = label_map[prediction]

        print(
            "Predicted user:",
            predicted_name
        )

    else:

        print(
            "Unknown user"
        )

In [ ]:
CONFIDENCE_THRESHOLD = 70

def recognize_face(face_image):

    prediction, confidence = recognizer.predict(
        face_image
    )

    if (
        prediction in label_map
        and confidence < CONFIDENCE_THRESHOLD
    ):

        name = label_map[prediction]

        return True, name, confidence

    return False, "Unknown", confidence

In [ ]:
for (x, y, w, h) in faces_detected:

    face = gray[
        y:y+h,
        x:x+w
    ]

    authorized, name, confidence = recognize_face(
        face
    )

    print()
    print("----------------------------")

    if authorized:

        print("ACCESS GRANTED")
        print("User:", name)
        print("Confidence:", confidence)

    else:

        print("ACCESS DENIED")
        print("User:", name)
        print("Confidence:", confidence)

    print("----------------------------")

In [ ]:
def process_face_authentication(image):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    detected_faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    results = []

    for (x, y, w, h) in detected_faces:

        face = gray[
            y:y+h,
            x:x+w
        ]

        authorized, name, confidence = recognize_face(
            face
        )

        results.append({
            "box": (x, y, w, h),
            "authorized": authorized,
            "name": name,
            "confidence": confidence
        })

    return results

In [ ]:
test_path = take_photo(
    "final_test.jpg"
)

image = cv2.imread(
    test_path
)

results = process_face_authentication(
    image
)

for result in results:

    x, y, w, h = result["box"]

    authorized = result["authorized"]
    name = result["name"]
    confidence = result["confidence"]

    if authorized:

        text = f"ACCESS GRANTED: {name}"

    else:

        text = "ACCESS DENIED"

    cv2.rectangle(
        image,
        (x, y),
        (x+w, y+h),
        (0, 255, 0) if authorized else (0, 0, 255),
        3
    )

    cv2.putText(
        image,
        text,
        (x, y-10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0) if authorized else (0, 0, 255),
        2
    )

    print(
        text,
        "| Distance:",
        round(confidence, 2)
    )

plt.figure(figsize=(10, 7))

plt.imshow(
    cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis("off")
plt.show()

In [ ]:
NEW_USER_NAME = "Persona Random"

NEW_USER_DIR = os.path.join(
    DATASET_DIR,
    NEW_USER_NAME
)

os.makedirs(
    NEW_USER_DIR,
    exist_ok=True
)

print(
    "New user folder created:",
    NEW_USER_DIR
)